# Day 005 — Queue · Merge Sort · Interpolation Search
**Date:** 2026-08-12  |  **Difficulty:** Intermediate  |  **Series:** Daily DSA

---

## What You Will Learn Today
| Topic | Concept | Time Complexity |
|-------|---------|----------------|
| Data Structure | **Queue** (FIFO) | Enqueue/Dequeue O(1) via `collections.deque` |
| Sorting | **Merge Sort** | Always O(n log n) — the gold standard divide-and-conquer sort |
| Searching | **Interpolation Search** | Best O(log log n), Worst O(n) — beats Binary on uniform data |

> **Series recap**
> - Day 001: Array · Bubble Sort · Linear Search
> - Day 002: Singly Linked List · Selection Sort · Jump Search O(√n)
> - Day 003: Doubly Linked List · Insertion Sort · Binary Search O(log n)
> - Day 004: Stack (LIFO) · Shell Sort · Sentinel Search
> - **Today**: Queue (FIFO) · Merge Sort O(n log n) · Interpolation Search O(log log n)

> Run each cell top-to-bottom with **Shift+Enter** to follow along interactively.

---
## PART 1 — Data Structure: Queue

A **Queue** is a **First-In, First-Out (FIFO)** collection.  
Think of a checkout line — whoever joins first, leaves first.

### Visual
```
Enqueue →  [ D | C | B | A ]  → Dequeue
           (rear)       (front)
```

### Stack vs Queue — the key distinction
| Property | Stack (LIFO) | Queue (FIFO) |
|----------|-------------|-------------|
| Add | push to **top** | enqueue at **rear** |
| Remove | pop from **top** | dequeue from **front** |
| Order | last in, **first out** | first in, **first out** |
| Use cases | undo, call stack, DFS | BFS, scheduling, print spooler |

### Why `collections.deque` and not a plain list?
- `list.pop(0)` is **O(n)** — every element shifts left.
- `deque.popleft()` is **O(1)** — backed by a doubly-linked list of fixed-size blocks.

### Queue variants
| Variant | Description |
|---------|-------------|
| Simple Queue | standard FIFO |
| Circular Queue | fixed capacity, wraps around |
| Deque (double-ended) | enqueue/dequeue at both ends |
| Priority Queue | dequeues highest-priority first |

In [ ]:
# ─── Queue Implementation ─────────────────────────────────────────────────────
from collections import deque

class Queue:
    """
    FIFO Queue backed by collections.deque.
    enqueue and dequeue are both O(1).
    """

    def __init__(self, max_size=None):
        self._data     = deque()
        self._max_size = max_size

    def enqueue(self, item):
        """Add item to the rear — O(1)."""
        if self._max_size and len(self._data) >= self._max_size:
            raise OverflowError(f'Queue full (max={self._max_size})')
        self._data.append(item)         # append to right (rear)

    def dequeue(self):
        """Remove and return the front item — O(1)."""
        if self.is_empty():
            raise IndexError('dequeue from empty queue')
        return self._data.popleft()     # pop from left (front)

    def peek(self):
        """Return the front item without removing it — O(1)."""
        if self.is_empty():
            raise IndexError('peek at empty queue')
        return self._data[0]

    def is_empty(self):  return len(self._data) == 0
    def size(self):      return len(self._data)

    def __repr__(self):
        if self.is_empty():
            return 'Queue(empty)'
        items = ' → '.join(str(x) for x in self._data)
        return f'Queue(front [{items}] rear)'


# ── Demo ──────────────────────────────────────────────────────────────────────
q = Queue()
print("--- Enqueueing customers A, B, C, D ---")
for customer in ['Alice', 'Bob', 'Carol', 'Dave']:
    q.enqueue(customer)
    print(f"  enqueue({customer!r:6}) → {q}")

print(f"\npeek()  → {q.peek()!r}  (front of queue, not removed)")
print(f"size()  → {q.size()}")

print("\n--- Dequeueing (FIFO order) ---")
while not q.is_empty():
    print(f"  dequeue() → {q.dequeue()!r:6}  | {q}")

In [ ]:
# ─── Classic Queue Application: BFS Level-Order Traversal ────────────────────

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val   = val
        self.left  = left
        self.right = right

def bfs_level_order(root):
    """
    Breadth-First Search using a Queue.
    Returns nodes level by level — impossible to do correctly with a Stack.

    Time: O(n)   Space: O(w) where w = max width of the tree
    """
    if root is None:
        return []

    result = []
    q = Queue()
    q.enqueue(root)

    while not q.is_empty():
        level_size = q.size()          # snapshot: how many nodes on this level
        level = []

        for _ in range(level_size):
            node = q.dequeue()
            level.append(node.val)
            if node.left:  q.enqueue(node.left)
            if node.right: q.enqueue(node.right)

        result.append(level)

    return result


#        1
#       / \
#      2   3
#     / \ / \
#    4  5 6  7
tree = TreeNode(1,
    TreeNode(2, TreeNode(4), TreeNode(5)),
    TreeNode(3, TreeNode(6), TreeNode(7))
)

levels = bfs_level_order(tree)
print("BFS Level-Order Traversal:")
for i, level in enumerate(levels):
    print(f"  Level {i}: {level}")

---
## PART 2 — Sorting Algorithm: Merge Sort

### Core Idea
**Divide** the array in half recursively until each sub-array has one element (trivially sorted).  
**Conquer** by merging pairs of sorted sub-arrays into a larger sorted array.  
Merge is the key step: compare the fronts of two sorted halves and take the smaller element.

### Step-by-step on `[38, 27, 43, 3, 9, 82, 10]`
```
Divide:
  [38,27,43,3,9,82,10]
  [38,27,43]       [3,9,82,10]
  [38] [27,43]     [3,9]  [82,10]
       [27][43]    [3][9] [82][10]

Conquer (merge):
       [27,43]     [3,9]  [10,82]
  [27,38,43]       [3,9,10,82]
  [3,9,10,27,38,43,82]  ✓
```

### Why Merge Sort is important
- **Guaranteed O(n log n)** — no worst-case degradation unlike Quick Sort.
- **Stable** — equal elements preserve their original order.
- **Parallelisable** — independent sub-problems map naturally to multiple threads.
- **External sort** — used to sort data larger than RAM (merge sorted file chunks).
- Python's `timsort` (used by `sorted()` and `.sort()`) is a hybrid of Merge Sort + Insertion Sort.

### Complexity
| Case | Time | Space |
|------|------|-------|
| Best | O(n log n) | O(n) |
| Average | O(n log n) | O(n) |
| Worst | O(n log n) | O(n) |

> The O(n) space is the price for stability and guaranteed performance — in-place merge sort exists but is complex and slower in practice.

In [ ]:
# ─── Merge Sort ───────────────────────────────────────────────────────────────

def merge_sort(arr, depth=0, verbose=False):
    """
    Sort `arr` using Merge Sort (top-down, recursive).

    Returns a new sorted list (does not mutate the input).

    Args:
        arr     : list of comparable elements
        depth   : recursion depth (used only for verbose indentation)
        verbose : print divide and merge steps
    """
    indent = '  ' * depth

    if len(arr) <= 1:
        if verbose: print(f"{indent}base case: {arr}")
        return arr[:]

    mid   = len(arr) // 2
    left  = arr[:mid]
    right = arr[mid:]

    if verbose: print(f"{indent}divide {arr} → {left} | {right}")

    left_sorted  = merge_sort(left,  depth + 1, verbose)
    right_sorted = merge_sort(right, depth + 1, verbose)

    merged = _merge(left_sorted, right_sorted)

    if verbose: print(f"{indent}merge  {left_sorted} + {right_sorted} → {merged}")

    return merged


def _merge(left, right):
    """
    Merge two sorted lists into one sorted list — O(n).
    The core of Merge Sort: compare front elements, take the smaller.
    """
    result = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:    # <= preserves stability
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    # Append any remaining elements (at most one side has leftovers)
    result.extend(left[i:])
    result.extend(right[j:])
    return result

In [ ]:
# ─── Verbose walkthrough ──────────────────────────────────────────────────────
sample = [38, 27, 43, 3]
print(f"Input: {sample}\n")
sorted_arr = merge_sort(sample, verbose=True)
print(f"\nResult: {sorted_arr}")

In [ ]:
# ─── Test Cases ───────────────────────────────────────────────────────────────
test_cases = [
    ([38, 27, 43, 3, 9, 82, 10],  "random"),
    ([1, 2, 3, 4, 5],             "already sorted"),
    ([5, 4, 3, 2, 1],             "reverse sorted"),
    ([42],                        "single element"),
    ([],                          "empty list"),
    ([3, 3, 1, 1, 2],             "with duplicates"),
    ([-4, 0, 7, -1, 3],           "with negatives"),
    (list(range(15, 0, -1)),      "reverse 1-15"),
]

print(f"{'Input':<36} {'Sorted':<36} {'Case'}")
print("-" * 85)
for data, label in test_cases:
    result = merge_sort(data)
    print(f"{str(data):<36} {str(result):<36} {label}")

---
## PART 3 — Searching Algorithm: Interpolation Search

### Core Idea
Binary Search always checks the **middle** index.  
Interpolation Search is smarter — it **estimates where the target likely is** based on its value relative to the range, like a human would in a phone book.

### The probe formula
```
pos = lo + ((target - arr[lo]) * (hi - lo)) / (arr[hi] - arr[lo])
```
For a **uniformly distributed** sorted array this is like a linear interpolation — the probe lands very close to the target immediately.

### Step-by-step on `[10, 20, 30, 40, 50, 60, 70, 80, 90, 100]`, target = 70
```
lo=0, hi=9, arr[lo]=10, arr[hi]=100
pos = 0 + ((70-10) * (9-0)) / (100-10)
    = 0 + (60 * 9) / 90
    = 0 + 6  = 6
arr[6] = 70  → FOUND in 1 step!
```
Binary Search would need 3–4 steps for the same array.

### When to use Interpolation Search
- Array is **sorted AND uniformly distributed** → O(log log n) expected
- Array is sorted but **non-uniform** → degrades toward O(n) in the worst case
- If distribution is unknown → Binary Search is safer

### Complexity
| Case | Time | Space |
|------|------|-------|
| Best (uniform, target present) | O(1) | O(1) |
| Average (uniform distribution) | O(log log n) | O(1) |
| Worst (non-uniform / exponential) | O(n) | O(1) |

In [ ]:
# ─── Interpolation Search ─────────────────────────────────────────────────────

def interpolation_search(arr, target, verbose=False):
    """
    Search for `target` in a SORTED array using Interpolation Search.
    Works best on uniformly distributed data.

    Args:
        arr     : sorted list of comparable numbers
        target  : value to find
        verbose : print each probe step if True

    Returns:
        int : index of target, or -1 if not found
    """
    lo, hi = 0, len(arr) - 1
    step = 0

    while lo <= hi and arr[lo] <= target <= arr[hi]:
        step += 1

        # Guard: avoid division by zero when all remaining elements are equal
        if arr[lo] == arr[hi]:
            if arr[lo] == target:
                if verbose: print(f"  Step {step}: all equal, found at {lo}")
                return lo
            break

        # Interpolation probe — estimate where target sits in [lo, hi]
        pos = lo + int((target - arr[lo]) * (hi - lo) / (arr[hi] - arr[lo]))

        if verbose:
            print(f"  Step {step}: lo={lo}({arr[lo]}) hi={hi}({arr[hi]}) "
                  f"probe={pos}({arr[pos]})")

        if arr[pos] == target:
            if verbose: print(f"  → FOUND at index {pos}")
            return pos
        elif arr[pos] < target:
            if verbose: print(f"  → {arr[pos]} < {target}, search right")
            lo = pos + 1
        else:
            if verbose: print(f"  → {arr[pos]} > {target}, search left")
            hi = pos - 1

    if verbose: print(f"  → NOT FOUND after {step} steps")
    return -1

In [ ]:
# ─── Verbose walkthrough ──────────────────────────────────────────────────────
uniform = list(range(10, 110, 10))   # [10,20,...,100]
print(f"Array (uniform): {uniform}\n")

print("Search for 70 (uniform — hits on step 1):")
interpolation_search(uniform, 70, verbose=True)

print("\nSearch for 30:")
interpolation_search(uniform, 30, verbose=True)

print("\nSearch for 55 (absent):")
interpolation_search(uniform, 55, verbose=True)

In [ ]:
# ─── Test Cases ───────────────────────────────────────────────────────────────
search_tests = [
    (list(range(10,110,10)), 70,  "uniform — found (1 step)"),
    (list(range(10,110,10)), 10,  "found at start"),
    (list(range(10,110,10)), 100, "found at end"),
    (list(range(10,110,10)), 55,  "not found (between elements)"),
    (list(range(10,110,10)), 200, "not found (beyond range)"),
    ([42],                    42,  "single element — found"),
    ([42],                    1,   "single element — not found"),
    ([],                      5,   "empty list"),
    (list(range(0,1000,3)),   501, "333-element uniform range"),
    ([1, 2, 4, 8, 16, 32],   16,  "exponential (non-uniform) — still works"),
]

print(f"{'Array (preview)':<32} {'Target':>7}  {'Result':<14} {'Case'}")
print("-" * 82)
for arr, target, label in search_tests:
    result = interpolation_search(arr, target)
    preview = str(arr[:4]) + ('...' if len(arr) > 4 else '')
    found   = f"index {result}" if result != -1 else "not found"
    print(f"{preview:<32} {str(target):>7}  {found:<14} {label}")

---
## PART 4 — Putting It All Together

Real-world pipeline: **task scheduler**
1. Tasks arrive and are queued in FIFO order (**Queue**)
2. At batch time, all pending tasks are sorted by priority (**Merge Sort**)
3. A specific priority level is located instantly (**Interpolation Search**)

In [ ]:
# ─── End-to-End Example ───────────────────────────────────────────────────────
import random
random.seed(5)

# 1. Tasks arrive in random order — enqueue each
task_queue = Queue()
task_names = [f"task_{i:02d}" for i in range(1, 11)]
priorities  = random.sample(range(10, 110, 10), 10)  # uniform: 10,20,...,100

print("1. Tasks arriving (FIFO queue):")
for name, prio in zip(task_names, priorities):
    task_queue.enqueue((prio, name))
    print(f"   enqueue priority={prio:>3}  {name}")

# 2. Drain queue into a list, sort by priority with Merge Sort
pending = []
while not task_queue.is_empty():
    pending.append(task_queue.dequeue())

print(f"\n2. Pending tasks (arrival order): {[p for p,_ in pending]}")

sorted_tasks = merge_sort([p for p, _ in pending])
print(f"   After Merge Sort (priorities) : {sorted_tasks}")

# 3. Interpolation Search for a specific priority
target_prio = sorted_tasks[4]   # pick the 5th-highest priority
pos = interpolation_search(sorted_tasks, target_prio)
print(f"\n3. Interpolation Search for priority {target_prio}:")
print(f"   Found at sorted index {pos}")

# 4. Summary
print(f"\n--- Scheduler Summary ---")
print(f"   Total tasks     : {len(sorted_tasks)}")
print(f"   Highest priority: {sorted_tasks[-1]}")
print(f"   Lowest priority : {sorted_tasks[0]}")
print(f"   Execution order : {sorted_tasks[::-1]}")

---
## Complexity Cheat Sheet

```
┌──────────────────────────────┬──────────┬──────────┬──────────┬─────────┐
│ Operation                    │ Best     │ Average  │ Worst    │ Space   │
├──────────────────────────────┼──────────┼──────────┼──────────┼─────────┤
│ Queue enqueue / dequeue      │ O(1)     │ O(1)     │ O(1)     │ O(n)   │
│ Queue peek / is_empty        │ O(1)     │ O(1)     │ O(1)     │ O(1)   │
├──────────────────────────────┼──────────┼──────────┼──────────┼─────────┤
│ Merge Sort                   │ O(n log n│ O(n log n│ O(n log n│ O(n)   │
│   (comparisons)              │ O(n/2)   │ O(n log n│ O(n log n│        │
├──────────────────────────────┼──────────┼──────────┼──────────┼─────────┤
│ Interpolation Search (uniform│ O(1)     │ O(log lgn│ O(n)     │ O(1)   │
│ Binary Search (any sorted)   │ O(1)     │ O(log n) │ O(log n) │ O(1)   │
└──────────────────────────────┴──────────┴──────────┴──────────┴─────────┘
```

## Key Takeaways
- Queue's **FIFO discipline** is essential for BFS, task schedulers, and any system where order of arrival matters — `collections.deque` gives true O(1) dequeue vs list's O(n).
- Merge Sort's **guaranteed O(n log n)** and stability make it the foundation of production sort algorithms (Timsort). The O(n) space cost is the only trade-off.
- Interpolation Search's **O(log log n)** is theoretically better than Binary Search's O(log n) — but only on uniformly distributed data; use Binary Search when uncertain about the distribution.

---
**Tomorrow — Day 006:** Circular Queue · Quick Sort · Fibonacci Search